
# Clustering K-means di moduli software

## Obiettivo

In questo notebook analizziamo un dataset sintetico che descrive **160 moduli software** attraverso alcune metriche tecniche.

L'obiettivo è mostrare, in modo didattico, come usare **K-means** per individuare gruppi di moduli con caratteristiche simili **senza disporre di una variabile target**.

Le feature disponibili sono:

| Feature | Significato |
|---|---|
| `LOC` | Lines Of Code: numero di righe di codice |
| `CyclomaticComplexity` | Complessità ciclomatica |
| `FanIn` | Numero di moduli che dipendono dal modulo osservato |
| `FanOut` | Numero di moduli da cui dipende il modulo osservato |
| `Commits_6m` | Numero di commit negli ultimi 6 mesi |
| `Bugs_6m` | Numero di bug registrati negli ultimi 6 mesi |
| `TestCoverage` | Percentuale di copertura dei test |

La colonna `Module` è un **identificatore** e quindi **non deve essere usata da K-means**.

---

## Idea fondamentale

K-means non conosce etichette come:

- "modulo semplice";
- "modulo core";
- "modulo instabile";
- "modulo legacy".

L'algoritmo vede soltanto numeri e cerca gruppi di osservazioni vicine nello spazio delle feature.

Sarà poi compito nostro **interpretare i cluster** analizzandone i centroidi e le caratteristiche medie.


# Contenuti (non è l'indice)

- caricamento e controllo del dataset;
- analisi dei valori mancanti e duplicati;
- statistiche descrittive;
- istogrammi;
- boxplot;
- matrice di correlazione;
- scatter plot tra metriche significative;
- spiegazione della standardizzazione;
- StandardScaler;
- scelta di K con Elbow Method;
- scelta di K con Silhouette Score;
- applicazione di K-means con K=4;
- numerosità dei cluster;
- profilo medio dei cluster;
- analisi dei centroidi standardizzati;
- interpretazione dei cluster;
- visualizzazioni 2D;
- visualizzazione mediante PCA;
- elenco dei moduli appartenenti ai diversi cluster;
- valutazione finale con Silhouette Score;
- conclusioni e spunti di discussione per sviluppatori.

In [ ]:

# Librerie utilizzate nel notebook

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# Impostazioni generali per una visualizzazione più leggibile
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True



# 1. Caricamento del dataset

Il notebook assume che il file:

```text
kmeans_moduli_software.csv
```

si trovi **nella stessa cartella del notebook**.


In [ ]:

# Percorso del dataset: stessa cartella del notebook
DATA_PATH = Path("kmeans_moduli_software.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"File non trovato: {DATA_PATH.resolve()}\n"
        "Copiare kmeans_moduli_software.csv nella stessa cartella del notebook."
    )

df = pd.read_csv(DATA_PATH)

print(f"Numero di righe:   {df.shape[0]}")
print(f"Numero di colonne: {df.shape[1]}")

df.head(10)



# 2. Prima esplorazione dei dati

Prima di applicare qualsiasi algoritmo di Machine Learning è buona pratica capire:

- quali colonne sono presenti;
- quali sono numeriche;
- se esistono valori mancanti;
- se ci sono righe duplicate;
- quali sono gli ordini di grandezza delle variabili.


In [ ]:

# Tipi delle colonne e numerosità dei valori non nulli
df.info()


In [ ]:

# Controllo dei valori mancanti
missing = df.isna().sum().to_frame("Valori_mancanti")
missing["Percentuale"] = 100 * missing["Valori_mancanti"] / len(df)

missing


In [ ]:

# Controllo dei duplicati
n_duplicates = df.duplicated().sum()

print(f"Righe duplicate: {n_duplicates}")



## Statistiche descrittive

`describe()` permette di osservare rapidamente:

- media;
- deviazione standard;
- minimo e massimo;
- quartili.

È già possibile notare che le feature hanno **scale molto diverse**.

Per esempio:

- `LOC` può essere nell'ordine delle migliaia;
- `Bugs_6m` nell'ordine delle unità o decine;
- `TestCoverage` è una percentuale.

Questo punto sarà fondamentale prima di applicare K-means.


In [ ]:

numeric_cols = [
    "LOC",
    "CyclomaticComplexity",
    "FanIn",
    "FanOut",
    "Commits_6m",
    "Bugs_6m",
    "TestCoverage"
]

df[numeric_cols].describe().T



# 3. Distribuzione delle feature

Osserviamo la distribuzione delle variabili numeriche.

Gli istogrammi aiutano a capire:

- il range dei valori;
- eventuali asimmetrie;
- la presenza di gruppi o concentrazioni;
- possibili valori estremi.


In [ ]:

# Istogrammi delle feature numeriche

n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 10))
axes = axes.flatten()

for ax, col in zip(axes, numeric_cols):
    ax.hist(df[col], bins=18, edgecolor="black", alpha=0.75)
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("Frequenza")

# Nasconde eventuali riquadri non utilizzati
for ax in axes[len(numeric_cols):]:
    ax.axis("off")

fig.suptitle("Distribuzione delle metriche software", fontsize=16)
plt.tight_layout()
plt.show()



# 4. Boxplot e possibili valori estremi

I boxplot sono utili per confrontare la dispersione e individuare osservazioni potenzialmente anomale.

Qui visualizziamo ogni variabile separatamente perché le scale sono molto diverse.


In [ ]:

for col in numeric_cols:
    plt.figure(figsize=(8, 2.5))
    plt.boxplot(df[col], vert=False)
    plt.title(f"Boxplot - {col}")
    plt.xlabel(col)
    plt.show()



# 5. Correlazioni tra le metriche

La matrice di correlazione misura la relazione lineare tra coppie di variabili.

Valori vicini a:

- **+1** → forte relazione lineare positiva;
- **0** → debole relazione lineare;
- **-1** → forte relazione lineare negativa.

Attenzione: correlazione non significa causalità.

Nel nostro contesto può essere interessante verificare, per esempio, se:

- moduli più grandi tendono a essere più complessi;
- maggiore complessità si accompagna a più bug;
- maggiore copertura dei test si associa a meno bug.


In [ ]:

corr = df[numeric_cols].corr()

corr.round(2)


In [ ]:

# Heatmap della matrice di correlazione realizzata con Matplotlib

fig, ax = plt.subplots(figsize=(9, 7))

im = ax.imshow(corr.values, aspect="auto")

ax.set_xticks(range(len(numeric_cols)))
ax.set_xticklabels(numeric_cols, rotation=45, ha="right")

ax.set_yticks(range(len(numeric_cols)))
ax.set_yticklabels(numeric_cols)

# Scriviamo il coefficiente dentro ogni cella
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        ax.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

fig.colorbar(im, ax=ax, label="Correlazione")
ax.set_title("Matrice di correlazione")
plt.tight_layout()
plt.show()



# 6. Alcune relazioni interessanti tra coppie di feature

Prima del clustering possiamo osservare alcune proiezioni bidimensionali.

Ricordiamo però che K-means lavorerà contemporaneamente su **tutte le feature selezionate**, non soltanto su due.


In [ ]:

# Complessità vs LOC
plt.figure(figsize=(8, 5))
plt.scatter(df["LOC"], df["CyclomaticComplexity"], alpha=0.7)
plt.xlabel("LOC")
plt.ylabel("CyclomaticComplexity")
plt.title("Dimensione del modulo vs complessità ciclomatica")
plt.show()


In [ ]:

# Bug vs Test Coverage
plt.figure(figsize=(8, 5))
plt.scatter(df["Bugs_6m"], df["TestCoverage"], alpha=0.7)
plt.xlabel("Bugs_6m")
plt.ylabel("TestCoverage")
plt.title("Bug negli ultimi 6 mesi vs copertura dei test")
plt.show()


In [ ]:

# FanIn vs FanOut
plt.figure(figsize=(8, 5))
plt.scatter(df["FanIn"], df["FanOut"], alpha=0.7)
plt.xlabel("FanIn")
plt.ylabel("FanOut")
plt.title("Dipendenze in ingresso vs dipendenze in uscita")
plt.show()



# 7. Preparazione dei dati per K-means

## Escludiamo `Module`

`Module` è semplicemente il nome del modulo software.

Non rappresenta una quantità misurabile e quindi non entra nel calcolo delle distanze.

## Perché dobbiamo standardizzare?

K-means usa la **distanza euclidea**.

Se lasciassimo i dati originali:

```text
LOC = 6000
Bugs_6m = 12
FanOut = 30
TestCoverage = 65
```

`LOC`, essendo numericamente molto più grande, avrebbe un peso sproporzionato nel calcolo delle distanze.

Per evitare questo problema utilizziamo `StandardScaler`.

Dopo la standardizzazione, ogni feature ha approssimativamente:

- media = 0;
- deviazione standard = 1.

In questo modo le feature contribuiscono in maniera più equilibrata alla distanza.


In [ ]:

# Matrice delle feature utilizzate dal modello
X = df[numeric_cols].copy()

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Convertiamo il risultato in DataFrame solo per poterlo ispezionare più facilmente
X_scaled_df = pd.DataFrame(X_scaled, columns=numeric_cols)

X_scaled_df.head()


In [ ]:

# Verifica didattica: media circa 0 e deviazione standard circa 1

pd.DataFrame({
    "Media": X_scaled_df.mean(),
    "Deviazione_standard": X_scaled_df.std(ddof=0)
}).round(3)



# 8. Quanti cluster scegliere?

K-means richiede di specificare in anticipo il numero di cluster `K`.

Non esiste sempre un unico valore "corretto".

Useremo due strumenti molto comuni:

1. **Elbow Method**
2. **Silhouette Score**

---

## Elbow Method

Per ogni valore di `K` calcoliamo l'**inertia**:

> somma delle distanze quadratiche delle osservazioni dal centroide del proprio cluster.

All'aumentare di K l'inertia diminuisce sempre.

Cerchiamo quindi il punto in cui il miglioramento comincia a diventare molto più piccolo: il cosiddetto **gomito**.


In [ ]:

k_values = range(2, 9)

inertias = []
silhouettes = []

for k in k_values:
    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = model.fit_predict(X_scaled)

    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Numero di cluster K")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(list(k_values))
plt.show()



## Silhouette Score

La silhouette misura quanto ogni osservazione è:

- vicina agli altri punti del proprio cluster;
- lontana dai punti degli altri cluster.

Il valore varia approssimativamente tra **-1 e +1**.

In generale:

- vicino a **1** → cluster ben separati;
- vicino a **0** → cluster sovrapposti;
- negativo → alcune osservazioni potrebbero essere assegnate al cluster sbagliato.

Il valore di K con silhouette più alta è spesso un buon candidato, ma va sempre interpretato insieme al contesto.


In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), silhouettes, marker="o")
plt.xlabel("Numero di cluster K")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score per diversi valori di K")
plt.xticks(list(k_values))
plt.show()

best_k = list(k_values)[int(np.argmax(silhouettes))]

print(f"K con silhouette più alta: {best_k}")
print(f"Silhouette massima: {max(silhouettes):.3f}")



# 9. Applichiamo K-means

Per questa esercitazione utilizziamo:

```text
K = 4
```

Il dataset è stato costruito in modo da contenere alcuni profili software distinguibili, ma K-means **non conosce questi profili**.

Il parametro:

```python
random_state=42
```

serve a rendere riproducibile il risultato.

`n_init=20` fa eseguire K-means più volte con inizializzazioni differenti e conserva la soluzione migliore.


In [ ]:

K = 4

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=20
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

df.head(10)



# 10. Quanti moduli ci sono in ciascun cluster?


In [ ]:

cluster_sizes = (
    df["Cluster"]
    .value_counts()
    .sort_index()
    .rename("Numero_moduli")
    .to_frame()
)

cluster_sizes


In [ ]:

plt.figure(figsize=(7, 4))
plt.bar(
    cluster_sizes.index.astype(str),
    cluster_sizes["Numero_moduli"]
)
plt.xlabel("Cluster")
plt.ylabel("Numero di moduli")
plt.title("Dimensione dei cluster")
plt.show()



# 11. Profilo dei cluster

Il numero del cluster (`0`, `1`, `2`, `3`) **non ha alcun significato semantico**.

`Cluster 0` non significa, per esempio, "cluster migliore".

Per capire cosa rappresentano i gruppi dobbiamo osservare i valori medi delle feature nei diversi cluster.


In [ ]:

cluster_profile = (
    df
    .groupby("Cluster")[numeric_cols]
    .mean()
    .round(1)
)

cluster_profile



## Confronto con la media complessiva

Un modo molto utile per interpretare i cluster è confrontare ogni centroide con la media generale del dataset.

Usiamo i centroidi nello spazio standardizzato:

- valore **positivo** → sopra la media;
- valore **negativo** → sotto la media;
- valore vicino a **0** → vicino alla media generale.


In [ ]:

centroids_z = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=numeric_cols,
    index=[f"Cluster {i}" for i in range(K)]
)

centroids_z.round(2)


In [ ]:

# Visualizzazione dei centroidi standardizzati

fig, ax = plt.subplots(figsize=(11, 5))

x = np.arange(len(numeric_cols))
width = 0.18

for i in range(K):
    ax.bar(
        x + (i - (K - 1) / 2) * width,
        centroids_z.iloc[i],
        width=width,
        label=f"Cluster {i}"
    )

ax.axhline(0, linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(numeric_cols, rotation=45, ha="right")
ax.set_ylabel("Valore standardizzato del centroide")
ax.set_title("Profilo dei centroidi rispetto alla media generale")
ax.legend()

plt.tight_layout()
plt.show()



# 12. Interpretazione dei risultati

Osservando la tabella dei valori medi e i centroidi standardizzati, possiamo assegnare **etichette descrittive** ai cluster.

L'operazione importante è questa:

```text
K-means trova gruppi numerici
          ↓
noi osserviamo i centroidi
          ↓
noi interpretiamo il significato dei gruppi
```

Le etichette non sono prodotte dall'algoritmo.

Per esempio, un cluster potrebbe essere caratterizzato da:

- poche LOC;
- bassa complessità;
- pochi bug;
- alta test coverage.

Potremmo allora descriverlo come:

> **moduli piccoli, stabili e ben testati**

Un altro potrebbe avere:

- molte LOC;
- alta complessità;
- molti commit;
- molti bug;
- bassa coverage.

Potremmo descriverlo come:

> **moduli complessi e potenzialmente critici**

Il significato finale dipende sempre dal dominio e dagli obiettivi dell'analisi.


In [ ]:

# Tabelle ordinate per alcune metriche utili all'interpretazione

print("Cluster ordinati per LOC media:")
display(cluster_profile.sort_values("LOC"))

print("\nCluster ordinati per numero medio di bug:")
display(cluster_profile.sort_values("Bugs_6m"))

print("\nCluster ordinati per Test Coverage media:")
display(cluster_profile.sort_values("TestCoverage", ascending=False))



# 13. Visualizzazione bidimensionale dei cluster

K-means ha lavorato su **7 dimensioni**.

Per visualizzare il risultato su un piano utilizziamo prima alcune coppie di feature.

I colori rappresentano il cluster assegnato da K-means.


In [ ]:

plt.figure(figsize=(8, 5))

scatter = plt.scatter(
    df["LOC"],
    df["CyclomaticComplexity"],
    c=df["Cluster"],
    alpha=0.75
)

plt.xlabel("LOC")
plt.ylabel("CyclomaticComplexity")
plt.title("Cluster K-means: LOC vs complessità")
plt.colorbar(scatter, label="Cluster")
plt.show()


In [ ]:

plt.figure(figsize=(8, 5))

scatter = plt.scatter(
    df["Bugs_6m"],
    df["TestCoverage"],
    c=df["Cluster"],
    alpha=0.75
)

plt.xlabel("Bugs_6m")
plt.ylabel("TestCoverage")
plt.title("Cluster K-means: bug vs test coverage")
plt.colorbar(scatter, label="Cluster")
plt.show()



# 14. Visualizzazione con PCA

Per rappresentare in 2D l'informazione contenuta nelle **7 feature standardizzate**, possiamo utilizzare la PCA.

La PCA non viene utilizzata qui per creare i cluster.

La usiamo soltanto **dopo il clustering**, come strumento di visualizzazione.

Le due componenti principali sono combinazioni lineari delle feature originali che catturano la maggiore quantità possibile di variabilità.


In [ ]:

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

print(
    "Varianza spiegata dalle prime due componenti:",
    f"{pca.explained_variance_ratio_.sum():.1%}"
)

plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["Cluster"],
    alpha=0.8
)

plt.xlabel("Prima componente principale")
plt.ylabel("Seconda componente principale")
plt.title("Visualizzazione 2D dei cluster tramite PCA")
plt.colorbar(scatter, label="Cluster")
plt.show()



# 15. Quali moduli appartengono ai diversi cluster?

Ora possiamo vedere concretamente quali moduli sono stati raggruppati insieme.


In [ ]:

for cluster in sorted(df["Cluster"].unique()):
    print(f"\n=== CLUSTER {cluster} ===")

    display(
        df.loc[
            df["Cluster"] == cluster,
            ["Module"] + numeric_cols
        ]
        .sort_values("LOC", ascending=False)
        .head(10)
    )



# 16. Quanto è buona la soluzione con K = 4?

Calcoliamo il Silhouette Score della soluzione finale.

Ricordiamo che si tratta di una misura geometrica della separazione dei cluster, non di una misura di "correttezza" rispetto a etichette reali.


In [ ]:

final_silhouette = silhouette_score(X_scaled, df["Cluster"])

print(f"Silhouette Score con K={K}: {final_silhouette:.3f}")



# 17. Cosa abbiamo imparato

## Pipeline completa

```text
Dataset
   ↓
Analisi esplorativa
   ↓
Selezione delle feature numeriche
   ↓
Standardizzazione
   ↓
Scelta di K
   ↓
K-means
   ↓
Assegnazione dei cluster
   ↓
Analisi dei centroidi
   ↓
Interpretazione dei cluster
```

## Messaggi chiave

### 1. K-means è un algoritmo non supervisionato

Non esiste una variabile target.

### 2. K-means raggruppa osservazioni simili

La nozione di "simile" deriva dalla distanza tra le osservazioni nello spazio delle feature.

### 3. La scala delle variabili è fondamentale

Con feature su scale diverse è normalmente necessario standardizzare i dati.

### 4. K deve essere scelto

Elbow Method e Silhouette Score sono strumenti utili, ma la scelta deve anche avere senso nel dominio applicativo.

### 5. I numeri dei cluster non significano nulla

`Cluster 0`, `Cluster 1`, ecc. sono soltanto identificatori.

### 6. L'interpretazione è compito dell'analista

K-means trova la struttura numerica; il significato business o tecnico deve essere attribuito successivamente.

---

# Possibile discussione con un team di sviluppo

A questo punto possiamo porre alcune domande:

- I moduli con molti bug e bassa test coverage meritano una priorità di refactoring?
- I moduli con alto `FanIn` sono componenti core da proteggere con più test?
- I moduli con molti commit sono in evoluzione oppure instabili?
- Un modulo molto complesso ma con alta copertura dei test rappresenta lo stesso rischio di uno poco testato?
- Quali altre metriche software potremmo aggiungere?

Alcune possibili feature aggiuntive:

- technical debt;
- code churn;
- numero di sviluppatori che hanno modificato il modulo;
- numero di incidenti in produzione;
- tempo medio di risoluzione dei bug;
- numero di dipendenze esterne;
- vulnerabilità di sicurezza;
- età del modulo.

Questa è la parte più importante dell'uso reale del clustering: **trasformare gruppi matematici in informazioni utili per prendere decisioni**.
